# Reporte de Hallazgos, Riesgos y Recomendaciones
**Proyecto:** Predicción de Rotación de Empleados (Attrition)  
**Sprint:** 5 — Evaluation  
**Rol:** Documentation Lead (PB-18)  
**Dataset:** Employee Performance & Attrition (14,000 registros)

## Resumen Ejecutivo

El equipo desarrolló un pipeline completo de Machine Learning para predecir la rotación de empleados (attrition) en una organización. A lo largo de 4 sprints se exploraron los datos, se construyeron features, se entrenaron modelos baseline y se aplicaron técnicas de ensamble y tuning.

**Resultado final:** El modelo ensamblado (Stacking Classifier) obtuvo un AUC-ROC de ~0.48 en el test set, equivalente a una predicción aleatoria. Este resultado **no cumple con los criterios de éxito definidos en el Sprint 1** y se recomienda detener el pase a producción hasta incorporar nuevas variables con mayor poder predictivo.

## Riesgos Identificados

A continuación se documentan los principales riesgos identificados durante el desarrollo del proyecto, siguiendo el patrón: **Hallazgo → Evidencia → Impacto → Recomendación → Próximos pasos**.

## Riesgo 1: Señal Predictiva Insuficiente en los Datos

**Hallazgo:** El modelo final (Stacking Classifier) obtuvo un AUC-ROC de ~0.48 en el test set, equivalente a una predicción aleatoria, a pesar de haber aplicado técnicas avanzadas de ensamble y tuning con Optuna.

**Evidencia:** `14_final_validation.ipynb` — AUC-ROC: ~0.48, IC 95%: límite superior < 0.50. F1-Score Macro < 0.50.

**Impacto:** Las variables disponibles en el dataset no contienen suficiente señal para predecir la rotación de empleados de forma confiable. Cualquier predicción del modelo actual sería equivalente a una decisión aleatoria.

**Recomendación:** Revisar la estrategia de modelado — explorar modelos más simples e interpretables que puedan capturar mejor la señal disponible, y profundizar en el análisis exploratorio para identificar qué variables tienen mayor correlación real con attrition.

**Próximos pasos:** El equipo debe realizar un análisis de importancia de variables (feature importance) sobre los modelos baseline para identificar qué variables aportan mayor información.

## Riesgo 2: Desbalance de Clases

**Hallazgo:** El dataset presenta un fuerte desbalance entre clases — solo el 16% de empleados rotaron (clase 1) frente al 84% que no rotaron (clase 0). Este desbalance provocó que los modelos baseline sin balanceo obtuvieran Recall=0.00 para la clase de interés.

**Evidencia:** `07_class_balance.ipynb` — Distribución: 11,840 registros clase 0 vs 2,160 registros clase 1 (ratio 5.5:1). Modelos sin SMOTE: Recall clase 1 = 0.00.

**Impacto:** El desbalance genera modelos que predicen siempre "No rota" obteniendo 84% de accuracy de forma engañosa. Esto hace que la métrica de accuracy no sea representativa del desempeño real del modelo para detectar empleados en riesgo.

**Recomendación:** Mantener SMOTE correctamente integrado dentro del pipeline de cross-validation. Evaluar adicionalmente el uso de class_weight='balanced' como alternativa más conservadora. Usar siempre Recall y AUC-ROC como métricas principales en lugar de accuracy.

**Próximos pasos:** El Pipeline Builder debe verificar que el balanceo esté correctamente integrado antes de cualquier nuevo experimento. El equipo debe reportar siempre métricas separadas por clase.

## Riesgo 3: Feature Engineering Limitado

**Hallazgo:** El equipo creó únicamente 3 features derivadas (ratio_salario_edad, antiguedad_satisfaccion, rango_edad), todas basadas en transformaciones simples de variables ya existentes en el dataset.

**Evidencia:** `06_feature_eng.ipynb` — Dataset pasó de 20 a 23 columnas. Las features creadas son un ratio, un producto y un binning de variables existentes.

**Impacto:** El modelo no aprovechó todo el potencial informativo del dataset disponible. Variables como overtime_hours_monthly, work_life_balance_score y job_satisfaction podrían generar interacciones con mayor poder predictivo que no fueron exploradas.

**Recomendación:** Explorar combinaciones no lineales entre variables existentes, aplicar transformaciones logarítmicas a variables asimétricas, y crear features de interacción entre satisfacción laboral, horas extra y distancia al trabajo — todo dentro del mismo dataset académico disponible.

**Próximos pasos:** El Feature Engineer debe proponer al menos 5 nuevas features derivadas de las variables existentes y validar su impacto en AUC-ROC mediante cross-validation.

## Riesgo 4: Inconsistencia Metodológica entre Notebooks

**Hallazgo:** Cada notebook recreó el preprocesamiento de forma diferente — algunos usaron el pipeline guardado (preprocessing_pipeline.pkl), otros aplicaron LabelEncoder manual, y otros escalaron directamente con StandardScaler sin usar el pipeline oficial.

**Evidencia:** `08_pipeline.ipynb` construye y guarda el pipeline oficial. Sin embargo, `09_baseline_models.ipynb` lo carga correctamente mientras que `10_evaluation.ipynb` aplicó preprocesamiento manual independiente, generando resultados inconsistentes entre sprints.

**Impacto:** Las métricas obtenidas en distintos notebooks no son comparables entre sí, ya que los datos de entrada a los modelos fueron transformados de manera diferente. Esto dificulta identificar si una mejora en métricas se debe al modelo o al preprocesamiento aplicado.

**Recomendación:** Establecer como regla de equipo que todos los notebooks deben cargar y usar exclusivamente preprocessing_pipeline.pkl para cualquier transformación de datos. Nunca aplicar preprocesamiento manual fuera del pipeline oficial.

**Próximos pasos:** El Pipeline Builder debe documentar claramente en el README cómo cargar y usar el pipeline oficial. Todos los miembros del equipo deben revisar sus notebooks y corregir cualquier preprocesamiento manual.

## Riesgo 5: Posible Data Leakage en la Evaluación

**Hallazgo:** Durante el Sprint 3, los modelos baseline entrenados con SMOTE dentro del pipeline de cross-validation obtuvieron métricas muy altas (Random Forest AUC-ROC: 0.9662, KNN Recall: 0.9866). Sin embargo, el modelo final del Sprint 4 obtuvo AUC-ROC ~0.48 en el test set. Esta brecha tan grande sugiere que las métricas del Sprint 3 podrían haber sido infladas por data leakage.

**Evidencia:** `11_model_comparison.ipynb` — Random Forest AUC-ROC: 0.9662 en CV. `14_final_validation.ipynb` — Stacking Classifier AUC-ROC: ~0.48 en test set. Diferencia de ~0.48 puntos entre CV y test final.

**Impacto:** Si las métricas de cross-validation fueron infladas por leakage, las decisiones de selección de modelos tomadas en el Sprint 3 y Sprint 4 podrían haber sido incorrectas, llevando al equipo a invertir esfuerzo en modelos que en realidad no tenían capacidad predictiva real.

**Recomendación:** Verificar que SMOTE se aplique únicamente dentro de cada fold del cross-validation y nunca antes del split. Usar imblearn.pipeline.Pipeline en lugar de sklearn.pipeline.Pipeline para garantizar que SMOTE solo actúe durante el entrenamiento de cada fold.

**Próximos pasos:** El Pipeline Builder debe auditar todos los notebooks del Sprint 3 y Sprint 4 para confirmar que no hubo leakage. Documentar los resultados.

## Riesgo 6: Limitaciones de Generalización del Modelo

**Hallazgo:** El modelo fue entrenado sobre un dataset académico de 14,000 registros con características específicas de una organización simulada. Las variables disponibles (satisfacción laboral, salario, horas extra, entre otras) pueden no reflejar la complejidad real de los factores que influyen en la rotación de empleados en una organización real.

**Evidencia:** `02_data_loading.ipynb` — Dataset de 14,000 registros con 20 variables. El dataset fue proporcionado con fines académicos y no proviene de un entorno productivo real. Distribución de attrition: 16% clase positiva.

**Impacto:** Un modelo entrenado en este dataset no puede garantizar su desempeño en datos reales de una organización, ya que las distribuciones, patrones y variables disponibles en producción podrían diferir significativamente de los datos de entrenamiento.

**Recomendación:** Tratar los resultados de este proyecto como un ejercicio metodológico y no como un modelo listo para producción. En un contexto real, sería necesario recolectar datos propios de la organización, validar la calidad de los datos y realizar un análisis de deriva (drift) periódico.

**Próximos pasos:** El equipo debe documentar claramente en el reporte ejecutivo que este modelo fue desarrollado con fines académicos y que su despliegue en producción requeriría una validación exhaustiva con datos reales.

## Conclusión y Decisión

### Veredicto Técnico

El modelo final desarrollado a lo largo de los 4 sprints **no cumple con los criterios de éxito definidos en el Sprint 1**. Un AUC-ROC de ~0.48 indica que el modelo no tiene capacidad real de discriminación entre empleados que rotarán y los que no.

### Decisión

**Se recomienda detener el pase a producción** del modelo actual y retomar el proyecto desde la fase de Feature Engineering con las siguientes acciones prioritarias:

| Prioridad | Acción | Responsable |
|---|---|---|
| Alta | Auditar data leakage en pipelines del Sprint 3 y 4 | Pipeline Builder |
| Alta | Explorar nuevas features derivadas de variables existentes | Feature Engineer |
| Media | Estandarizar uso del pipeline oficial en todos los notebooks | Todo el equipo |
| Media | Validar integración correcta de SMOTE en cross-validation | Pipeline Builder |
| Baja | Documentar limitaciones académicas del dataset | Documentation Lead |

### Lección Aprendida

La arquitectura computacional y algorítmica del proyecto es robusta — el equipo aplicó correctamente SMOTE, cross-validation estratificado, tuning con Optuna y ensambles. El problema raíz es la **ausencia de señal predictiva suficiente** en las variables disponibles del dataset académico, lo cual no puede resolverse con técnicas de modelado más complejas sino con mejores features.